In [2]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="eymenslimani/plant-disease-detector",
    filename="best_model.pth"
)

print(model_path)

KeyboardInterrupt: 

In [ ]:
class_names = [
    "Apple_Scab_Leaf",
    "Apple_leaf",
    "Apple_rust_leaf",
    "Bell_pepper_leaf",
    "Bell_pepper_leaf_spot",
    "Blueberry_leaf",
    "Cherry_leaf",
    "Corn_Gray_leaf_spot",
    "Corn_leaf_blight",
    "Corn_rust_leaf",
    "Peach_leaf",
    "Potato_leaf_early_blight",
    "Potato_leaf_late_blight",
    "Raspberry_leaf",
    "Soyabean_leaf",
    "Squash_Powdery_mildew_leaf",
    "Strawberry_leaf",
    "Tomato_Early_blight_leaf",
    "Tomato_Septoria_leaf_spot",
    "Tomato_leaf",
    "Tomato_leaf_bacterial_spot",
    "Tomato_leaf_late_blight",
    "Tomato_leaf_mosaic_virus",
    "Tomato_leaf_yellow_virus",
    "Tomato_mold_leaf",
    "Tomato_two_spotted_spider_mites_leaf",
    "grape_leaf",
    "grape_leaf_black_rot",
]

In [ ]:
checkpoint = torch.load(
    model_path,
    map_location="cuda",
    weights_only=False
)

print(type(checkpoint))

if isinstance(checkpoint, dict):
    print(checkpoint.keys())

<class 'dict'>
dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_acc', 'test_acc', 'val_f1', 'test_f1'])


In [ ]:
import timm
import torch

model = timm.create_model(
    "tf_efficientnetv2_m.in21k_ft_in1k",
    pretrained=False,
    num_classes=28
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [ ]:
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np

transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    ToTensorV2()
])

image = Image.open("/content/internet_tomato.jpg").convert("RGB")

image_tensor = transform(
    image=np.array(image)
)["image"].unsqueeze(0)

In [ ]:
with torch.no_grad():
    logits = model(image_tensor)
    probs = torch.softmax(logits, dim=1)

pred_idx = probs.argmax(dim=1).item()
confidence = probs[0, pred_idx].item()

print("Class:", class_names[pred_idx])
print("Confidence:", f"{confidence:.2%}")

Class: Tomato_leaf_mosaic_virus
Confidence: 22.85%
